In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --- Step 1: Load and Inspect the Data ---
# For this example, we'll create a sample DataFrame.
# In a real project, you would load it from a CSV file: df = pd.read_csv('loan_data.csv')
data = {
    'Gender': ['Male', 'Male', 'Female', 'Male', 'Female', 'Male', 'Male', 'Female', None, 'Male'],
    'Married': ['Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes'],
    'Credit_History': [1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, None],
    'ApplicantIncome': [5849, 4583, 3000, 2583, 6000, 5417, 2333, 3036, 4006, 12841],
    'LoanAmount': [128, 128, 66, 120, 141, 267, 95, 158, 168, 349],
    'Loan_Status': ['Y', 'N', 'Y', 'Y', 'N', 'Y', 'N', 'N', 'Y', 'N']
}
df = pd.DataFrame(data)

print("--- Initial Data ---")
print(df.head())
print("\n--- Data Info ---")
df.info()

# --- Step 2: Data Preprocessing & Cleaning ---
# A. Handle missing values (imputation)
# For simplicity, we'll fill missing categorical data with the mode and numerical data with the median.
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].median(), inplace=True)

# B. Convert categorical columns to numerical data using LabelEncoder
# scikit-learn models require all input features to be numeric.
le = LabelEncoder()
for col in ['Gender', 'Married', 'Loan_Status']:
    df[col] = le.fit_transform(df[col])

# After encoding, 'Y' becomes 1 and 'N' becomes 0 in 'Loan_Status'
# 'Male' becomes 1, 'Female' becomes 0
# 'Yes' becomes 1, 'No' becomes 0

print("\n--- Preprocessed Data ---")
print(df.head())

# --- Step 3: Define Features (X) and Target (y) ---
# We select the features that we think are important for the prediction.
features = ['Gender', 'Married', 'Credit_History', 'ApplicantIncome', 'LoanAmount']
X = df[features]
y = df['Loan_Status']

# --- Step 4: Split Data into Training and Testing Sets ---
# We'll use 80% of the data for training and 20% for testing the model's performance.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# --- Step 5: Train the Logistic Regression Model ---
# Initialize the model
model = LogisticRegression(random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)
print("\n✅ Model training complete!")

# --- Step 6: Evaluate the Model ---
# Make predictions on the unseen test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

# Display a more detailed performance report
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)']))

print("\n--- Confusion Matrix ---")
# Shows True Positives, False Positives, True Negatives, False Negatives
print(confusion_matrix(y_test, y_pred))


# --- Step 7: Make a Prediction on New Data ---
# Let's create a new applicant profile to see what the model predicts.
# [Gender(Male=1), Married(Yes=1), Credit_History(Yes=1.0), ApplicantIncome, LoanAmount]
new_applicant = [[1, 1, 1.0, 5000, 150]]
prediction = model.predict(new_applicant)
proba_prediction = model.predict_proba(new_applicant)

# The output of le.inverse_transform needs a list/array
status_map = {1: 'Approved (Y)', 0: 'Rejected (N)'}
result = status_map[prediction[0]]

print(f"\n--- New Applicant Prediction ---")
print(f"Prediction for new applicant: **{result}**")
print(f"Prediction Probability [Rejected, Approved]: {proba_prediction}")